# 🎙️ استوديو «رِواق» لاستنساخ الصوت وإنتاج الحلقات (Google Colab المجاني — T4 GPU)

أهلاً بك في الاستوديو الصوتي الرسمي لمشروع **«رِواق»** — بديل مجاني ومفتوح المصدر بالكامل لـ ElevenLabs، مربوط مباشرة بسلسلة الماسترينج المعتمدة (`voice_finish.py` · `LUFS -16` · `48kHz`) ومولّد كابشنز `episode_meta.json`.

### ⚡ قبل البدء (تأكد من خطوة واحدة فقط):
من القائمة العلوية في Colab اختر: **`Runtime` ⬅️ `Change runtime type` ⬅️ `T4 GPU` ⬅️ `Save`**.

---
### 🗺️ خريطة الاستخدام السريعة:
- **أول مرة فقط (إعداد الموديل الخاص بصوتك):** شغّل الخلايا **1 ⬅️ 2 ⬅️ 3** بالترتيب.
- **في كل حلقة جديدة يومياً (بدون إعادة تدريب):** شغّل الخلية **1** ثم الخلية **4** مباشرة لتفتح لك الواجهة الرسومية بالعربي!

In [ ]:
#@title 1️⃣ ربط Google Drive وتثبيت بيئة استوديو «رِواق» (تُشغَّل دائماً في بداية كل جلسة — ~90 ثانية)
import os, shutil, subprocess, sys
from pathlib import Path

DRIVE_FOLDER_NAME = "Riwaq_Voice_Lab" #@param {type:"string"}
INSTALL_XTTS_ENGINE = True #@param {type:"boolean"}
REPO_URL = "https://github.com/omarretco27/my-voice-cloning.git" #@param {type:"string"}
REPO_BRANCH = "arena/a44deae5-my-voice-cloning" #@param {type:"string"}

# 1. التحقق من وجود كارت الشاشة T4 GPU
import torch
if not torch.cuda.is_available():
    print("⚠️ تنبيه هام: لم يتم تفعيل كارت الشاشة GPU! اذهب إلى Runtime -> Change runtime type واختر T4 GPU.")
else:
    print(f"✅ كارت الشاشة جاهز: {torch.cuda.get_device_name(0)}")

# 2. ربط Google Drive وتكوين هيكل المجلدات القياسي
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}"
DRIVE_REPO_CACHE = f"{DRIVE_ROOT}/_riwaq_repo"
WORK_REPO = "/content/riwaq-system"

for sub in ["01_raw_recordings", "02_golden_reference", "03_clean_dataset", "04_models", "05_episodes_out", "06_reports"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

# 3. سحب سكريبتات رِواق وحفظ نسخة احتياطية على Google Drive للعمل حتى لو أصبح المستودع Private لاحقاً
if os.path.exists(WORK_REPO):
    shutil.rmtree(WORK_REPO)

cloned = False
for br in [REPO_BRANCH, "main"]:
    res = subprocess.run(["git", "clone", "--depth", "1", "-b", br, REPO_URL, WORK_REPO], capture_output=True, text=True)
    if res.returncode == 0 and os.path.exists(f"{WORK_REPO}/colab/riwaq_colab_app.py"):
        cloned = True
        print(f"✅ تم سحب أحدث نسخة من نظام رِواق (فرع: {br})")
        if os.path.exists(DRIVE_REPO_CACHE):
            shutil.rmtree(DRIVE_REPO_CACHE)
        shutil.copytree(WORK_REPO, DRIVE_REPO_CACHE, ignore=shutil.ignore_patterns(".git", "node_modules"))
        break

if not cloned:
    if os.path.exists(f"{DRIVE_REPO_CACHE}/colab/riwaq_colab_app.py"):
        shutil.copytree(DRIVE_REPO_CACHE, WORK_REPO)
        print("✅ تم تحميل نظام رِواق من النسخة المحفوظة على Google Drive مباشرة.")
    else:
        raise RuntimeError("تعذر الوصول للمستودع. تأكد من الرابط أو انسخ مجلد المشروع إلى _riwaq_repo على Drive.")

# 4. تحميل محرك Applio (RVC v2 الحديث المتوافق مع Python 3.10-3.12 بدون مشاكل fairseq)
APPLIO_DIR = "/content/Applio"
if not os.path.exists(APPLIO_DIR):
    print("⏳ جاري تحميل محرك البصمة الصوتية RVC v2 (Applio)...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/IAHispano/Applio.git", APPLIO_DIR], check=True)

# 5. تثبيت الحزم الصوتية اللازمة مع الحفاظ على PyTorch الخاص بـ Colab لسرعة التثبيت
print("⏳ جاري تثبيت حزم معالجة الصوت والذكاء الاصطناعي...")
subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libsndfile1"], check=True)

base_pkgs = [
    "faiss-cpu", "librosa>=0.11.0", "soundfile", "stftpitchshift", "soxr",
    "torchcrepe", "torchfcpe", "einops", "onnx", "onnxruntime-gpu",
    "edge-tts", "webrtcvad-wheels", "noisereduce", "faster-whisper", "gradio"
]
if INSTALL_XTTS_ENGINE:
    base_pkgs.append("coqui-tts")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *base_pkgs], check=True)

os.environ["APPLIO_LOGS_DIR"] = f"{DRIVE_ROOT}/04_models"
os.environ["COQUI_TOS_AGREED"] = "1"
if f"{WORK_REPO}/colab" not in sys.path:
    sys.path.insert(0, f"{WORK_REPO}/colab")
if f"{WORK_REPO}/scripts" not in sys.path:
    sys.path.insert(0, f"{WORK_REPO}/scripts")

print("\n🎉 اكتمل التجهيز بنجاح!")
print(f"📂 مجلد مشروعك على Google Drive: {DRIVE_ROOT}")
print(f"   ├── 01_raw_recordings  (ضع تسجيلات صوتك الخام هنا)")
print(f"   ├── 02_golden_reference (ضع أنقى مقطع 15-25 ثانية هنا، أو سيختاره الكود تلقائياً)")
print(f"   ├── 04_models          (يُحفظ هنا موديل صوتك تلقائياً للأبد)")
print(f"   └── 05_episodes_out    (تخرج هنا حلقات رِواق الجاهزة للتحميل)")

---
### 🧹 الخطوة الثانية (تُشغَّل مرة واحدة فقط بعد رفع تسجيلاتك الخام إلى `01_raw_recordings`)
تأكد أنك رفعت ملفات صوتك بصيغة `WAV` أو `MP3` أو `M4A` أو حتى فيديوهات `MP4` داخل مجلد **`Riwaq_Voice_Lab/01_raw_recordings`** على Google Drive، ثم اضغط زر التشغيل بالأسفل.

In [ ]:
#@title 2️⃣ فحص وتنظيف وتقطيع تسجيلاتك الخام أوتوماتيك (تُشغَّل مرة واحدة عند رفع البيانات)
import sys
sys.path.insert(0, "/content/riwaq-system/scripts")
import prepare_dataset

DRIVE_FOLDER_NAME = "Riwaq_Voice_Lab" #@param {type:"string"}
ENABLE_GENTLE_DENOISE = True #@param {type:"boolean"}
MIN_CHUNK_SEC = 3.5 #@param {type:"slider", min:2.5, max:5.0, step:0.5}
MAX_CHUNK_SEC = 10.0 #@param {type:"slider", min:7.0, max:12.0, step:0.5}

DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}"
summary = prepare_dataset.run_pipeline(
    raw_dir=f"{DRIVE_ROOT}/01_raw_recordings",
    clean_dir=f"{DRIVE_ROOT}/03_clean_dataset",
    golden_dir=f"{DRIVE_ROOT}/02_golden_reference",
    report_path=f"{DRIVE_ROOT}/06_reports/DATASET_AUDIT_REPORT.md",
    min_sec=MIN_CHUNK_SEC,
    max_sec=MAX_CHUNK_SEC,
    enable_nr=ENABLE_GENTLE_DENOISE,
)
print(f"\n📊 الخلاصة الهندسية: {summary['verdict']}")

---
### 🧠 الخطوة الثالثة (تُشغَّل مرة واحدة فقط لتدريب بصمة صوت عمر `omar_el_clone_v1`)
- تحفظ هذه الخلية نسخة من الموديل على Google Drive **كل 25 Epoch تلقائياً**.
- **ميزة الأمان ضد فصل كولاب المجاني:** إذا فصل المتصفح أو انتهت الجلسة في منتصف التدريب، فقط أعد تشغيل الخلية **(1)** ثم الخلية **(3)** وسيكمل التدريب تلقائياً من آخر Epoch وصل إليه دون ضياع دقيقة واحدة!

In [ ]:
#@title 3️⃣ تدريب موديل بصمة الصوت (RVC v2 · 48kHz · RMVPE) مع الحفظ التلقائي على Google Drive
import sys
sys.path.insert(0, "/content/riwaq-system/colab")
import riwaq_colab_app

DRIVE_FOLDER_NAME = "Riwaq_Voice_Lab" #@param {type:"string"}
MODEL_NAME = "omar_el_clone_v1" #@param {type:"string"}
TOTAL_EPOCHS = 250 #@param {type:"slider", min:100, max:500, step:25}
SAVE_EVERY_EPOCH = 25 #@param {type:"slider", min:10, max:50, step:5}
BATCH_SIZE = 8 #@param {type:"slider", min:4, max:16, step:2}

DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}"
riwaq_colab_app.train_omar_rvc_model(
    drive_root=DRIVE_ROOT,
    applio_dir="/content/Applio",
    model_name=MODEL_NAME,
    total_epochs=int(TOTAL_EPOCHS),
    save_every_epoch=int(SAVE_EVERY_EPOCH),
    batch_size=int(BATCH_SIZE),
    sample_rate=48000,
)

---
### 🚀 الخطوة الرابعة — استوديو الإنتاج اليومي لحلقات «رِواق» (واجهة رسومية عربية بضغطة زر)
شغّل الخلية التالية ليظهر لك رابط **`https://....gradio.live`** وواجهة تفاعلية مباشرة تحتوي على:
1. **من السكريبت إلى حزمة الحلقة الكاملة:** تكتب نص المشاهد بالعامية المصرية ⬅️ يولّد `vo_01.wav, vo_02.wav...` بصوتك الماستر (`-16 LUFS` / `48kHz`) + كابشنز `Faster-Whisper` المضبوطة (`≤28 حرف`) + `episode_meta.json` في ملف `ZIP` واحد جاهز للريندر السحابي!
2. **من تسجيل موبايل سريع إلى صوت الاستوديو:** تسجل بصوتك من الموبايل في أي مكان ⬅️ ينظفه ويطبعه بخامة المايك الاحترافي وسلسلة استوديو رِواق!

In [ ]:
#@title 4️⃣ فتح واجهة استوديو «رِواق» التفاعلية (للإنتاج اليومي للحلقات)
import sys
sys.path.insert(0, "/content/riwaq-system/colab")
import riwaq_colab_app

DRIVE_FOLDER_NAME = "Riwaq_Voice_Lab" #@param {type:"string"}
MODEL_NAME = "omar_el_clone_v1" #@param {type:"string"}
CREATE_PUBLIC_LINK = True #@param {type:"boolean"}

DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}"
riwaq_colab_app.launch_riwaq_gradio_studio(
    drive_root=DRIVE_ROOT,
    applio_dir="/content/Applio",
    model_name=MODEL_NAME,
    share=CREATE_PUBLIC_LINK,
)